## Practice 3 Hugging Face Transformers: Sentiment Analysis and Binary Text Classification

**General Objective:**  
Become familiar with the Hugging Face Transformers ecosystem through two exercises:
- **Exercise 1:** Use an already fine-tuned model to perform inference.
- **Exercise 2:** Fine-tune a generic pretrained model for binary text classification.

### 1. Objective

- Distinguish between the two layers of knowledge: **Pretraining** (language knowledge) và **Downstream Fine-tuning**
  (sentiment classification knowledge).

| Exercise | Main Objective | Training Required? | Checkpoint Used |
|----------|----------------|--------------------|--------------------|
| **Exercise 1** | Inference + tokenizer understanding | Không | `distilbert-base-uncased-finetuned-sst-2-english` |
| **Exercise 2** | Full-process fine-tuning | Yes | `distilbert-base-uncased` (generic) |


### 2. Scientific and Theoretical Foundation

#### (a) Why Choose Rotten Tomatoes

| Criterion | Rotten Tomatoes | IMDb |
|---|---|---|
| Labeled samples | 10,662 | 50,000 |
| Train split | 8,530 | 25,000 |
| Validation split | 1,066 | No separate official split |
| Test split | 1,066 | 25,000 |
| Binary sentiment | Yes | Yes |
| Typical text length | Short | Longer |
| Fine-tuning cost | Lower | High hơn |
| Suitable for a compact practice notebook | Very high | High |
| Alignment with the official Hugging Face tutorial | Medium | Very high |

**Reason for choosing:** a binary classification task, a moderate computational workload for a practice notebook, three existing
train/validation/test splits, relatively short reviews, no need to create an additional validation split,
and support for a clean experimental workflow on a personal computer (CPU-only).

```
Train: 8,530
Validation: 1,066
Test: 1,066
Total: 10,662

Label 0: NEGATIVE
Label 1: POSITIVE
```

#### (b) Why Choose DistilBERT

```
flowchart LR
    A[Tokenized Text] --> B[DistilBERT Backbone]
    B --> C[Contextual Representation]
    C --> D[Classification Head]
    D --> E[Two Logits]
    E --> F[NEGATIVE or POSITIVE]
```

| Model | Main Advantages | Main Limitations |
|---|---|---|
| DistilBERT | Lightweight, practical | Lower capacity than BERT-base |
| BERT-base | Classic Transformer baseline | Higher computational cost |
| RoBERTa-base | Strong language representation | Higher computational cost |
| MiniLM | Very lightweight | Less aligned with the traditional introductory workflow |
| ALBERT | Parameter-efficient | Different architectural characteristics |

**Primary implementation choice: DistilBERT** - a pretrained Transformer that is lighter than BERT-base, directly supports
sequence classification, is suitable for English sentiment analysis, reduces training cost while preserving
the nature of the transfer-learning workflow, and is suitable for a CPU-only lab environment.

#### (c) Conceptual Nature of Transfer Learning

```
flowchart TD
    A[Pretraining] --> B[General Language Knowledge]
    B --> C[Downstream Fine-Tuning]
    C --> D[Binary Sentiment Knowledge]
    D --> E[Inference]
    E --> F[Positive or Negative Prediction]
```

```
Exercise 1
Already fine-tuned model
        |
        v
Inference

Exercise 2
Generic pretrained model
        |
        v
Task-specific fine-tuning
        |
        v
Binary classifier
```

#### (d) How Fine-tuning Differs from Feature Extraction

```
Feature extraction
Freeze Transformer backbone
        |
        v
Train only classifier

Fine-tuning
Update Transformer backbone
        +
Update classification head
```

| Criterion | Feature Extraction | Fine-tuning |
|----------|--------------------|-----------------------------------|
| Base model weights | Freeze (not updated) | Updated together with the classification head |
| Base model role | Only extracts fixed features | Adapts representations to fit the task |
| Computational cost | Lower | High hơn |
| Application in this practice | Not used | Yes (Trainer fine-tunes the entire model) |

Practice 3 (Exercise 2) applies **Fine-tuning** to the entire `distilbert-base-uncased` backbone, which is updated
together with the classification head; no layers are frozen.

#### (e) Things That Must Never Be Done Throughout Practice 3

```
Do not:
Fine-tune on the test set

Do not:
Use test performance to choose epochs or hyperparameters

Do not:
Aggressively remove stopwords, punctuation, or linguistic structure without justification

Do not:
Train DistilBERT from scratch for this practice

Do not:
Use an already sentiment-fine-tuned checkpoint in Exercise 2
and describe the process as generic pretrained-model fine-tuning
without explicitly documenting the checkpoint's prior task-specific training

Do not:
Fabricate loss curves, metrics, confusion matrices, or benchmark results
```

### 3. Overall End-to-End Pipeline Map

```
flowchart TD
    S[START] --> A[Environment and Seeds]

    A --> B[Exercise 1]
    B --> C[Load Fine-Tuned Sentiment Model]
    C --> D[Inspect Sentence Tokens]
    D --> E[Run Sentiment Inference]

    E --> F[Exercise 2]
    F --> G[Load Rotten Tomatoes Dataset]
    G --> H[EDA and Data Quality Checks]
    H --> I[Load DistilBERT Tokenizer]
    I --> J[Token-Length Analysis]
    J --> K[Tokenize Dataset]
    K --> L[Dynamic Padding]
    L --> M[Load Pretrained DistilBERT Classifier]
    M --> N[Model Sanity Check]
    N --> O[Define Metrics]
    O --> P[Configure TrainingArguments]
    P --> Q[Create Trainer]
    Q --> R[Debug Subset Run]
    R --> T{Pipeline Valid?}
    T -- No --> U[Fix and Re-run]
    U --> R
    T -- Yes --> V[Full Fine-Tuning]
    V --> W[Validation Monitoring]
    W --> X[Load Best Checkpoint]
    X --> Y[Final Test Evaluation]
    Y --> Z[Confusion Matrix]
    Z --> AA[Error Analysis]
    AA --> AB[New-Sentence Inference]
    AB --> AC[Save Model and Tokenizer]
    AC --> AD[Reload Sanity Test]
    AD --> AE[FINAL SUMMARY]
```

### 4. 16-Phase Diagram According to Notebook Architecture

```
flowchart TD
    P0[Phase 0<br/>Practice Overview] --> P1[Phase 1<br/>Environment and Reproducibility]
    P1 --> P2[Phase 2<br/>Pretrained Sentiment Inference]
    P2 --> P3[Phase 3<br/>Tokenization Investigation]
    P3 --> P4[Phase 4<br/>Dataset Loading]
    P4 --> P5[Phase 5<br/>EDA and Sanity Checks]
    P5 --> P6[Phase 6<br/>Tokenizer and Preprocessing]
    P6 --> P7[Phase 7<br/>Model Construction]
    P7 --> P8[Phase 8<br/>Metrics and Training Configuration]
    P8 --> P9[Phase 9<br/>Fine-Tuning]
    P9 --> P10[Phase 10<br/>Learning Curves]
    P10 --> P11[Phase 11<br/>Validation and Test Evaluation]
    P11 --> P12[Phase 12<br/>Error Analysis]
    P12 --> P13[Phase 13<br/>New-Sentence Inference]
    P13 --> P14[Phase 14<br/>Save and Reload]
    P14 --> P15[Phase 15<br/>Final Summary]
```

### 5. Stage 1 Scope

| Phase | Phase Name | Main Objective |
|-------|-----------|----------------|
| 0 | Practice Overview | Problem definition + academic architecture |
| 1 | Environment & Reproducibility | Set up environment, seed, device |
| 2 | Pretrained Sentiment Inference | Exercise 1 - Inference |
| 3 | Tokenization Investigation | Tokenizer analysis |
| 4 | Dataset Loading | Load Rotten Tomatoes |
| 5 | Dataset EDA & Sanity Checks | Data analysis |
| 6 | Tokenizer & Preprocessing | Prepare data for fine-tuning |


### 6. Key Technical Decisions

| Item | Decision | Status |
|----------|----------|--------|
| Model Exercise 1 | `distilbert-base-uncased-finetuned-sst-2-english` | Approved |
| Model Exercise 2 | `distilbert-base-uncased` | Approved |
| Dataset | Rotten Tomatoes | Approved |
| Hardware | CPU only | Approved |
| Random Seed | 42 | Approved |

### 7. Final Conceptual Summary

```
flowchart LR
    A[General Language Pretraining] --> B[Pretrained DistilBERT]
    B --> C[Binary Sentiment Fine-Tuning]
    C --> D[Task-Specific Classifier]
    D --> E[Validation]
    E --> F[Independent Test Evaluation]
    F --> G[Reusable Sentiment Model]
```

```
Exercise 1
Pretrained model reuse
        +
Tokenizer understanding

Exercise 2
Transfer learning
        +
Fine-tuning
        +
Scientific evaluation
        +
Reusable model artifacts
```

## Phase 1 Environment & Reproducibility

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root added: {project_root}") 

✅ Project root added: E:\KHDL\DeepLearning\practice 3\TOTAL_LABPARACTICE_FOR_DEEP_LEARNING\total_practice\practice_3


In [ ]:
# Phase 1: Environment & Reproducibility
from processing_own_phase.phase_01_environment import (
    set_seed,
    print_environment_info,
    save_environment_report,
)

# Set seed (must be done before running anything else)
set_seed(42)

# Print full report and save JSON file
info = print_environment_info()
filepath = save_environment_report(info)

# Quick summary
print(f"\nReport saved: {filepath}")
print(f"Device: {info['device']}")
print(f"Seed: {info['seed']}")


ENVIRONMENT INFORMATION
Timestamp: 2026-08-12T02:57:38.937502
Python Version: 3.11.9
Device: cpu
Global Seed: 42

--- Package Versions ---
❌ transformers    installed: MISSING      required: 5.14.1
❌ datasets        installed: MISSING      required: 5.0.1
❌ evaluate        installed: MISSING      required: 0.4.6
❌ accelerate      installed: MISSING      required: 1.14.0
⚠️ torch           installed: 2.13.0       required: 2.13.0+cpu
⚠️ numpy           installed: 2.4.2        required: 2.4.6
⚠️ matplotlib      installed: 3.11.0       required: 3.11.1
⚠️ pandas          installed: 3.0.1        required: 3.0.5
⚠️ scikit-learn    installed: 1.8.0        required: 1.9.0

--- Hugging Face Connectivity ---
❌ Status: failed
   Error: transformers is not installed

✅ Environment report saved to: docs\result\phase01-environment-log_2026-08-12_02-57-38.json

📁 Report saved: docs\result\phase01-environment-log_2026-08-12_02-57-38.json
💻 Device: cpu
🔢 Seed: 42


E:\KHDL\DeepLearning\practice 3\TOTAL_LABPARACTICE_FOR_DEEP_LEARNING\total_practice\practice_3\processing_own_phase\phase_01_environment.py:94: UserWarning: Package transformers is not installed. Required: 5.14.1
  warnings.warn(
E:\KHDL\DeepLearning\practice 3\TOTAL_LABPARACTICE_FOR_DEEP_LEARNING\total_practice\practice_3\processing_own_phase\phase_01_environment.py:94: UserWarning: Package datasets is not installed. Required: 5.0.1
  warnings.warn(
E:\KHDL\DeepLearning\practice 3\TOTAL_LABPARACTICE_FOR_DEEP_LEARNING\total_practice\practice_3\processing_own_phase\phase_01_environment.py:94: UserWarning: Package evaluate is not installed. Required: 0.4.6
  warnings.warn(
E:\KHDL\DeepLearning\practice 3\TOTAL_LABPARACTICE_FOR_DEEP_LEARNING\total_practice\practice_3\processing_own_phase\phase_01_environment.py:94: UserWarning: Package accelerate is not installed. Required: 1.14.0
  warnings.warn(
E:\KHDL\DeepLearning\practice 3\TOTAL_LABPARACTICE_FOR_DEEP_LEARNING\total_practice\practice

In [ ]:
# Phase 2: Pretrained Sentiment Inference
from processing_own_phase.phase_02_pretrained_inference import (
    get_sentiment_pipeline,
    run_inference,
    inspect_tokenizer
)

# Load pipeline (first time downloads ~260MB model, requires internet)
pipe = get_sentiment_pipeline()

# 3 sample sentences
test_sentences = [
    "I absolutely loved this movie! The performances were outstanding.",
    "This film was a complete waste of time. Terrible acting.",
    "It was okay, nothing special."
]

# Run inference
results = run_inference(pipe, test_sentences)
for sentence, result in zip(test_sentences, results):
    print(f"Sentence: {sentence[:60]}...")
    print(f"  Label: {result['label']}, Score: {result['score']:.4f}\n")

# Inspect tokenizer
token_info = inspect_tokenizer(test_sentences[0])
print("Tokenizer vocab size:", token_info['vocab_size'])
print("Tokens (first 20):", token_info['tokens'][:20])

In [ ]:
# Phase 3: Tokenization Investigation
from processing_own_phase.phase_03_tokenization import compare_tokenizers

# Compare tokenizers on the same sentence
sentence "I absolutely loved this movie! The performances were outstanding."
result = compare_tokenizers(sentence)

# Print conclusion
print(f"Are tokenizers identical? {result['are_identical']}")
print(f"\nVocab size (fine-tuned): {result['finetuned']['vocab_size']}")
print(f"Vocab size (base): {result['base']['vocab_size']}")

if result['are_identical']:
    print("\nCONCLUSION: The tokenizers from the fine-tuned model and the base model are IDENTICAL.")
    print("  -> Fine-tuning only updates the backbone model weights.")
    print("  -> The tokenizer remains unchanged, vocab_size = 30522.")
    print("  -> This confirms that the tokenizer does not change during fine-tuning.")
else:
    print("\nCONCLUSION: Tokenizers are different (unexpected).")

In [ ]:
# Phase 4: Dataset Loading
from processing_own_phase.phase_04_dataset_loading import (
    load_rotten_tomatoes,
    verify_split_sizes,  
    get_dataset_stats,
    check_null_values,
    print_sample_rows
)

# Load dataset
dataset = load_rotten_tomatoes()
print(f"Dataset loaded! Splits: {list(dataset.keys())}")

# Verify split sizes
actual_sizes = verify_split_sizes(dataset)
print(f"Split sizes verified: {actual_sizes}")

# Get statistics
stats = get_dataset_stats(dataset)
print("\nDataset Statistics:")
for split_name, split_stats in stats.items():
    print(f"\n{split_name.capitalize()}:")
    print(f"  Total: {split_stats['total']}")
    print(f"  Positive: {split_stats['positive']} ({split_stats['positive_pct']:.1f}%)")
    print(f"  Negative: {split_stats['negative']} ({split_stats['negative_pct']:.1f}%)")

# Check null values
null_report = check_null_values(dataset)
print("\nNull Values Check:")
all_clean = True
for split_name, null_counts in null_report.items():
    for col, count in null_counts.items():
        if count > 0:
            print(f"  {split_name}.{col}: {count} null values")
            all_clean = False
        else:
            print(f"  {split_name}.{col}: No null values")

if all_clean:
    print("\nDataset is clean! No null values found.")

# Print sample rows
print_sample_rows(dataset, "train", 3)

In [ ]:
# Phase 5: EDA and Sanity Checks
from processing_own_phase.phase_05_dataset_eda import (
    compute_text_lengths,
    plot_token_length_distribution,
    get_label_balance,
    recommend_max_length
)
from processing_own_phase.phase_06_preprocessing import get_tokenizer

# Load tokenizer base for token length computation
tokenizer = get_tokenizer()

# Compute length statistics (character, word, token)
stats = compute_text_lengths(dataset, tokenizer, "train")
print("Text Length Statistics (Train):")
for key, value in stats.items():
    print(f"  {key}:")
    for k, v in value.items():
        print(f"    {k}: {v:.2f}")

# Compute max_length based on P99
recommendation = recommend_max_length(dataset, tokenizer, "train")
print("\nRecommended max_length based on token distribution:")
for key, value in recommendation.items():
    print(f"  {key}: {value:.2f}")

# Plot and save histogram
plot_token_length_distribution(
    dataset, tokenizer, "train",
    max_length_reference=recommendation["p99"]
)

# Label balance
balance = get_label_balance(dataset)
print("\nLabel Balance:")
for split, data in balance.items():
    print(f"  {split}: Positive={data['positive']} ({data['positive_pct']:.1f}%), Negative={data['negative']} ({data['negative_pct']:.1f}%)")

# Final max_length decision
from config import MAX_TOKEN_LENGTH
print(f"\nDecision: max_length = {MAX_TOKEN_LENGTH} (based on P99 token length = {recommendation['p99']:.0f})")

In [ ]:
# Phase 6: Tokenizer and Preprocessing
from processing_own_phase.phase_06_preprocessing import (
    get_tokenizer,
    tokenize_dataset,
    show_tokenized_sample,
    get_data_collator
)

# Load tokenizer (base)
tokenizer = get_tokenizer()
print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Vocab size: {tokenizer.vocab_size}")

# Tokenize dataset
tokenized_dataset = tokenize_dataset(dataset, tokenizer)
print(f"Tokenized dataset splits: {list(tokenized_dataset.keys())}")

# Show a sample
show_tokenized_sample(tokenized_dataset, "train", 0)

# Check dynamic padding with DataCollator
collator = get_data_collator(tokenizer)
batch_samples = [tokenized_dataset["train"][i] for i in range(4)]
batch = collator(batch_samples)
print("\n--- Dynamic Padding Check ---")
print(f"Batch input_ids shape: {batch['input_ids'].shape}")
print(f"Batch attention_mask shape: {batch['attention_mask'].shape}")
print(f"Batch labels: {batch['labels'].tolist()}")
print("Dynamic padding works correctly!")

# Verify counts
print("\nFinal dataset sizes:")
for split in tokenized_dataset.keys():
    print(f"  {split}: {len(tokenized_dataset[split])} samples")

print("\nTokenized dataset ready for handoff to the next phase.")